In [1]:
# Standard libraries
import os
from pathlib import Path
from glob import glob

# Data handling
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr

# Visualization
import hvplot.pandas
import hvplot.xarray
import holoviews as hv
import matplotlib.pyplot as plt

# Mapping
import osmnx as ox
from shapely.geometry import Point

# EarthPy / AppEEARS
import earthpy
import earthpy.api.appeears as eaapp

In [2]:
project = earthpy.Project(
    "West Virginia Vegetation",
    dirname="west-virginia-vegetation"
)

project

In [3]:
lat = 39.631111
lon = -79.957199

study_point = Point(lon, lat)

study_area = gpd.GeoDataFrame(
    {"Site": ["West Virginia Study Area"]},
    geometry=[study_point],
    crs="EPSG:4326"
)

study_area

,Site,geometry
0,West Virginia Study Area,POINT (-79.9572 39.63111)


In [4]:
study_area = study_area.to_crs(3857)

study_area["geometry"] = study_area.buffer(500)

study_area = study_area.to_crs(4326)

study_area

,Site,geometry
0,West Virginia Study Area,"POLYGON ((-79.95271 39.63111, -79.95273 39.630..."


In [5]:
study_area.hvplot(
    geo=True,
    tiles="EsriImagery",
    alpha=0.4,
    line_color="red",
    width=700,
    height=500,
    title="West Virginia Study Area"
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [6]:
study_polygon = study_area.geometry.iloc[0]

nearby_buildings = ox.features_from_polygon(
    study_polygon,
    tags={"building": True}
)

nearby_buildings

geometry  \
element id                                                              
way     106844510   POLYGON ((-79.95923 39.62953, -79.95901 39.629...   
        106844514   POLYGON ((-79.95913 39.62963, -79.95891 39.629...   
        106846304   POLYGON ((-79.95507 39.6297, -79.95489 39.6296...   
        106846312   POLYGON ((-79.95813 39.62812, -79.95772 39.627...   
        106856454   POLYGON ((-79.95727 39.63268, -79.95727 39.632...   
...                                                               ...   
        1295100711  POLYGON ((-79.95754 39.63147, -79.95745 39.631...   
        1295100713  POLYGON ((-79.95672 39.63053, -79.95676 39.630...   
        1295101277  POLYGON ((-79.95401 39.63072, -79.95396 39.630...   
        1295101278  POLYGON ((-79.95408 39.63063, -79.95406 39.630...   
        1295101279  POLYGON ((-79.9542 39.63065, -79.95428 39.6305...   

                      building        source addr:housename     amenity  \
element id                                                                
way     106844510          yes          bing            NaN         NaN   
        106844514          yes  nearmap;Bing     Wing's Ole  restaurant   
        106846304   commercial          bing            NaN         NaN   
        106846312          yes          bing            NaN         NaN   
        106856454   university          bing            NaN         NaN   
...                        ...           ...            ...         ...   
        1295100711         yes           NaN            NaN         NaN   
        1295100713         yes           NaN            NaN         NaN   
        1295101277         yes           NaN            NaN         NaN   
        1295101278         yes           NaN            NaN         NaN   
        1295101279         yes           NaN            NaN         NaN   

                     cuisine                              name  \
element id                                                       
way     106844510        NaN                               NaN   
        106844514   american                        Wings Ole'   
        106846304        NaN                               NaN   
        106846312        NaN  Monongalia County Justice Center   
        106856454        NaN                        Knapp Hall   
...                      ...                               ...   
        1295100711       NaN                               NaN   
        1295100713       NaN                               NaN   
        1295101277       NaN                               NaN   
        1295101278       NaN                               NaN   
        1295101279       NaN                               NaN   

                                                        opening_hours height  \
element id                                                                     
way     106844510                                                 NaN    NaN   
        106844514   Mo-Th 11:00-24:00, Fr-Sa 11:00-02:00, Su 11:00...    NaN   
        106846304                                                 NaN    NaN   
        106846312                                                 NaN    NaN   
        106856454                                                 NaN   30.5   
...                                                               ...    ...   
        1295100711                                                NaN    NaN   
        1295100713                                                NaN    NaN   
        1295101277                                                NaN    NaN   
        1295101278                                                NaN    NaN   
        1295101279                                                NaN    NaN   

                   addr:city  ... outdoor_seating takeaway facebook leisure  \
element id                    ...                                             
way     106844510        NaN  ...             NaN      NaN      NaN     NaN  

In [7]:
study_area.hvplot(
    geo=True,
    tiles="EsriImagery",
    fill_alpha=0.2,
    line_color="red"
) * nearby_buildings.hvplot(
    geo=True,
    fill_alpha=0.5
)

:Overlay
   .WMTS.I      :WMTS   [Longitude,Latitude]
   .Polygons.I  :Polygons   [Longitude,Latitude]
   .Polygons.II :Polygons   [Longitude,Latitude]

In [8]:
ndvi_downloader = eaapp.AppeearsDownloader(
    download_key="wv_ndvi",
    project=project,
    product="MOD13Q1.061",
    layer="_250m_16_days_NDVI",
    start_date="07-01",
    end_date="07-31",
    recurring=True,
    year_range=[2017, 2023],   # Adjust if your event year differs
    polygon=study_area
)

In [9]:
ndvi_downloader.download_files(cache=False)

No stored credentials found for urs.earthdata.nasa.gov. Please log in.


/opt/conda/lib/python3.11/site-packages/earthpy/api/auth.py:192: UserWarning: Setting credentials not supported for 'netrc' backend.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/earthpy/api/auth.py:196: UserWarning: Failed to store credentials with 'keyring': No recommended backend was available. Install a recommended 3rd party backend package; or, install the keyrings.alt package if you want to use the non-recommended backends. See https://pypi.org/project/keyring for details.
  warnings.warn(


Credentials stored using 'env' backend.


In [10]:
ndvi_paths = sorted(
    glob(
        os.path.join(
            project.project_dir,
            "**",
            "*NDVI*.tif"
        ),
        recursive=True,
    )
)

print(f"Found {len(ndvi_paths)} NDVI files")

ndvi_paths[:5]

Found 21 NDVI files


['/workspaces/data/west-virginia-vegetation/wv_ndvi/MOD13Q1.061_2017167_to_2023212/MOD13Q1.061__250m_16_days_NDVI_doy2017177000000_aid0001.tif',
 '/workspaces/data/west-virginia-vegetation/wv_ndvi/MOD13Q1.061_2017167_to_2023212/MOD13Q1.061__250m_16_days_NDVI_doy2017193000000_aid0001.tif',
 '/workspaces/data/west-virginia-vegetation/wv_ndvi/MOD13Q1.061_2017167_to_2023212/MOD13Q1.061__250m_16_days_NDVI_doy2017209000000_aid0001.tif',
 '/workspaces/data/west-virginia-vegetation/wv_ndvi/MOD13Q1.061_2017167_to_2023212/MOD13Q1.061__250m_16_days_NDVI_doy2018177000000_aid0001.tif',
 '/workspaces/data/west-virginia-vegetation/wv_ndvi/MOD13Q1.061_2017167_to_2023212/MOD13Q1.061__250m_16_days_NDVI_doy2018193000000_aid0001.tif']

In [11]:
sample = rxr.open_rasterio(
    ndvi_paths[0],
    mask_and_scale=True
).squeeze()

sample

<xarray.DataArray (y: 4, x: 5)> Size: 80B
[20 values with dtype=float32]
Coordinates:
    band         int64 8B 1
  * x            (x) float64 40B -79.96 -79.96 -79.96 -79.96 -79.95
  * y            (y) float64 32B 39.63 39.63 39.63 39.63
    spatial_ref  int64 8B 0
Attributes:
    units:          NDVI
    AREA_OR_POINT:  Area

In [12]:
sample.hvplot(
    cmap="YlGn",
    rasterize=True,
    geo=True,
    title="Sample NDVI"
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'1e4d1509-a2d1-4689-8e05-795a6aa6c80f': {'version…

In [13]:
doy_start = -25
doy_end = -19

ndvi_das = []

for ndvi_path in ndvi_paths:

    ndvi_path = Path(ndvi_path)

    # Extract date from filename
    doy = ndvi_path.name[doy_start:doy_end]
    date = pd.to_datetime(doy, format="%Y%j")

    # Open raster
    da = rxr.open_rasterio(
        ndvi_path,
        mask_and_scale=True
    ).squeeze()

    # Add date coordinate
    da = da.assign_coords(date=date)
    da = da.expand_dims("date")
    da.name = "NDVI"

    ndvi_das.append(da)

len(ndvi_das)

21

In [14]:
ndvi_da = xr.combine_by_coords(
    ndvi_das,
    coords=["date"]
)

ndvi_da

/tmp/ipykernel_1369/605304272.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ndvi_da = xr.combine_by_coords(
/tmp/ipykernel_1369/605304272.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ndvi_da = xr.combine_by_coords(


<xarray.Dataset> Size: 2kB
Dimensions:      (date: 21, y: 4, x: 5)
Coordinates:
    band         int64 8B 1
  * x            (x) float64 40B -79.96 -79.96 -79.96 -79.96 -79.95
  * y            (y) float64 32B 39.63 39.63 39.63 39.63
    spatial_ref  int64 8B 0
  * date         (date) datetime64[ns] 168B 2017-01-17 2017-01-19 ... 2023-01-20
Data variables:
    NDVI         (date, y, x) float32 2kB 0.5362 0.5362 0.5362 ... 0.4777 0.5092

In [15]:
before = (
    ndvi_da
    .sel(date=slice("2017", "2019"))
    .mean("date")
    .NDVI
)

after = (
    ndvi_da
    .sel(date=slice("2021", "2023"))
    .mean("date")
    .NDVI
)

ndvi_diff = after - before

In [16]:
(
    ndvi_diff.hvplot(
        x="x",
        y="y",
        geo=True,
        cmap="PiYG",
        title="West Virginia NDVI Difference"
    )
    * study_area.hvplot(
        geo=True,
        fill_color=None,
        line_color="black"
    )
)

:Overlay
   .Image.I    :Image   [x,y]   (NDVI)
   .Polygons.I :Polygons   [Longitude,Latitude]